Data acquired from https://www.kaggle.com/datasets/jamiewelsh2/nba-per-game-player-statistics-2022-2023-season

In [1]:
import numpy as np
import pandas as pd

In [2]:
nba = pd.read_csv('nba_per_game_processed.csv')

In [3]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt

In [4]:
nba.loc[nba['Position']=='SG-PG', 'Position']='SG'
nba.loc[nba['Position']=='SF-SG', 'Position']='SF'
nba.loc[nba['Position']=='PG-SG', 'Position']='PG'
nba.loc[nba['Position']=='SF-PF', 'Position']='SF'
nba.loc[nba['Position']=='PF-C', 'Position']='PF'
nba.loc[nba['Position']=='PF-SF', 'Position']='PF'

In [5]:
nba['Position'].value_counts()

SG    128
C     108
SF    106
PF    104
PG     93
Name: Position, dtype: int64

In [6]:
nba.fillna(0, inplace = True)
nba.isna().sum()

Index                0
Player Name          0
Position             0
Age                  0
Team                 0
GP                   0
GS                   0
MP                   0
FG                   0
FGA                  0
FG%                  0
3P                   0
3PA                  0
3P%                  0
2P                   0
2PA                  0
2P%                  0
eFG%                 0
FT                   0
FTA                  0
FT%                  0
ORB                  0
DRB                  0
TRB                  0
AST                  0
STL                  0
BLK                  0
TOV                  0
PF                   0
PTS                  0
Player-additional    0
dtype: int64

In [7]:
nba = nba.drop(columns=['Index', 'Player Name', 'Age', '3P%', 'Team',
         'Player-additional', 'ORB', 'DRB', 'PTS', 'FG', '2P', '3P', 'FT', 'GS', 'FGA', '2PA', 'FG%', 'MP', 'TOV'])

In [8]:
nba.corr()

,GP,3PA,2P%,eFG%,FTA,FT%,TRB,AST,STL,BLK,PF
GP,1.000000,0.406479,0.198036,0.262181,0.392901,0.368063,0.486237,0.340165,0.414046,0.310445,0.514726
3PA,0.406479,1.000000,-0.030800,0.027551,0.446908,0.362148,0.203078,0.554130,0.467127,-0.031571,0.367351
2P%,0.198036,-0.030800,1.000000,0.736527,0.120541,0.176887,0.286196,-0.017440,-0.013446,0.250789,0.247026
eFG%,0.262181,0.027551,0.736527,1.000000,0.140700,0.129048,0.313491,0.037721,0.073195,0.252130,0.288132
FTA,0.392901,0.446908,0.120541,0.140700,1.000000,0.249940,0.618635,0.628892,0.472728,0.358970,0.514184
FT%,0.368063,0.362148,0.176887,0.129048,0.249940,1.000000,0.128492,0.246259,0.211434,0.055506,0.229070
TRB,0.486237,0.203078,0.286196,0.313491,0.618635,0.128492,1.000000,0.406246,0.407816,0.652032,0.713781
AST,0.340165,0.554130,-0.017440,0.037721,0.628892,0.246259,0.406246,1.000000,0.652689,0.089857,0.463605
STL,0.414046,0.467127,-0.013446,0.073195,0.472728,0.211434,0.407816,0.652689,1.000000,0.201542,0.530042
BLK,0.310445,-0.031571,0.250789,0.252130,0.358970,0.055506,0.652032,0.089857,0.201542,1.000000,0.520735


In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from statsmodels.tools.tools import add_constant
nba_vif = nba.drop('Position', axis = 1)
nba_vif = add_constant(nba_vif)
vif_df = pd.DataFrame()
vif_df['features'] = nba_vif.columns
vif_df['vif'] = [variance_inflation_factor(nba_vif.values, i)
                    for i in range(len(nba_vif.columns))]
vif_df

,features,vif
0,const,35.057788
1,GP,1.715740
2,3PA,1.851288
3,2P%,2.329729
4,eFG%,2.293315
5,FTA,2.384911
6,FT%,1.297943
7,TRB,3.428605
8,AST,2.600863
9,STL,2.066395


In [10]:
nba_r = nba.drop('Position', axis = 1)
nba.loc[0, 'Position']

'C'

In [11]:
from sklearn.model_selection import train_test_split
nba_train, nba_test = train_test_split(nba, test_size = 0.33, random_state = 42)

In [12]:
nba_new = nba_train.drop('Position', axis = 1)
nba_labels = nba_train['Position'].copy()

In [13]:
from sklearn.preprocessing import StandardScaler


In [15]:
# using liblinear because it is a small dataset
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipeline_logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(multi_class='ovr', solver='liblinear', max_iter=5000, random_state=42))
])
pipeline_logreg.fit(nba_new, nba_labels)

Pipeline(steps=[('scaler', StandardScaler()),
                ('log_reg',
                 LogisticRegression(max_iter=5000, multi_class='ovr',
                                    random_state=42, solver='liblinear'))])

In [17]:
pipeline_logreg.predict(nba_r.loc[[0]])

array(['C'], dtype=object)

In [18]:
from sklearn.model_selection import cross_val_score
(cross_val_score(pipeline_logreg, nba_new, nba_labels, cv=5, scoring='accuracy'))

array([0.60273973, 0.45833333, 0.47222222, 0.47222222, 0.58333333])

In [62]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())
display_scores((cross_val_score(pipeline_logreg, nba_new, nba_labels, cv=5, scoring='accuracy')))

Scores: [0.60273973 0.45833333 0.47222222 0.47222222 0.58333333]
Mean: 0.5177701674277018
Standard deviation: 0.06196825394383913


In [32]:
# trying to improve performance by clustering first
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
pipeline_km = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=45, random_state=42)),
    ('log_reg', LogisticRegression(multi_class='ovr', solver='liblinear', max_iter=5000, random_state=42))
])
pipeline_km.fit(nba_new, nba_labels)

Pipeline(steps=[('scaler', StandardScaler()),
                ('kmeans', KMeans(n_clusters=45, random_state=42)),
                ('log_reg',
                 LogisticRegression(max_iter=5000, multi_class='ovr',
                                    random_state=42, solver='liblinear'))])

In [33]:
cross_val_score(pipeline_km, nba_new, nba_labels, cv=5, scoring='accuracy')

array([0.52054795, 0.44444444, 0.55555556, 0.44444444, 0.59722222])

In [34]:
from sklearn.svm import SVC
pipeline_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(gamma='auto', random_state=42))
])
pipeline_svc.fit(nba_new, nba_labels)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svc', SVC(gamma='auto', random_state=42))])

In [36]:
cross_val_score(pipeline_svc, nba_new, nba_labels, cv=5, scoring='accuracy')

array([0.46575342, 0.47222222, 0.51388889, 0.375     , 0.54166667])

In [63]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())
display_scores(cross_val_score(pipeline_logreg, nba_new, nba_labels, cv=5, scoring='accuracy'))

Scores: [0.60273973 0.45833333 0.47222222 0.47222222 0.58333333]
Mean: 0.5177701674277018
Standard deviation: 0.06196825394383913


In [64]:
display_scores(cross_val_score(pipeline_km, nba_new, nba_labels, cv=5, scoring='accuracy'))

Scores: [0.52054795 0.44444444 0.55555556 0.44444444 0.59722222]
Mean: 0.5124429223744291
Standard deviation: 0.060596214764906654


In [65]:
display_scores(cross_val_score(pipeline_svc, nba_new, nba_labels, cv=5, scoring='accuracy'))

Scores: [0.46575342 0.47222222 0.51388889 0.375      0.54166667]
Mean: 0.4737062404870624
Standard deviation: 0.0566420978778449


In [37]:
nba_t = nba_test.drop('Position', axis=1)
nba_labels_test = nba_test['Position'].copy()

In [59]:
models = {'pipeline_logreg':pipeline_logreg, 'pipeline_km':pipeline_km, 'pipeline_svc':pipeline_svc}
for model in models:
    score = models[model].score(nba_t, nba_labels_test)
    print(f'{model}: {score}')

pipeline_logreg: 0.5393258426966292
pipeline_km: 0.5280898876404494
pipeline_svc: 0.5842696629213483


None of the models performed particularly well. Because of its performance on the test set, I am tempted to pick the SVC model, even though it had the worst mean cross validation score. 